# SunnyBest SFS — System Health Check

Run this notebook before any analysis session to verify:

| Check | What it verifies |
|-------|------------------|
| 1 | DB connection is alive |
| 2 | All expected tables exist in `core` schema |
| 3 | Row counts are non-zero and within expected range |
| 4 | Date freshness — most recent data is not stale |
| 5 | No nulls in critical columns |
| 6 | Price > cost for every product (margin sanity) |
| 7 | No negative units_sold or negative prices |
| 8 | Referential integrity — all fact FK values exist in dims |
| 9 | Promo discount_pct is bounded [0, 100] |
| 10 | Quick summary dashboard |

---
## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings("ignore")
pd.options.display.max_columns = 20
pd.options.display.width = 120

PALETTE = ["#4C72B0","#C44E52","#55A868","#DD8452","#8172B2","#64B5CD"]
STALE_DAYS = 7          # flag data older than this as stale
MIN_ROWS   = 100        # minimum expected rows per fact table

# ── DB connection ──────────────────────────────────────────
host     = "aws-1-eu-central-1.pooler.supabase.com"
port     = 5432
database = "postgres"
user     = "postgres.ogkdfmkybqtrsglcizzt"
password = quote_plus("YOUR_PASSWORD")

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}",
    pool_pre_ping=True
)

PASS = "\033[92m PASS\033[0m"
FAIL = "\033[91m FAIL\033[0m"
WARN = "\033[93m WARN\033[0m"

results = []   # collects (check_name, status, detail)

def record(name, passed, detail="", warn=False):
    status = "PASS" if passed else ("WARN" if warn else "FAIL")
    tag    = PASS   if passed else (WARN   if warn else FAIL)
    print(f"{tag}  {name}" + (f"  ->  {detail}" if detail else ""))
    results.append({"check": name, "status": status, "detail": detail})

print(f"Check started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## 1. DB Connection

In [ ]:
try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    record("DB connection", True, "postgresql+psycopg2 -> Supabase")
except Exception as e:
    record("DB connection", False, str(e))
    raise SystemExit("Cannot connect to DB -- aborting checks.")

---
## 2. Table Existence

In [ ]:
EXPECTED_TABLES = [
    "dim_products", "dim_stores", "dim_calendar", "dim_policy_regimes",
    "fact_sales", "fact_inventory", "fact_promotions",
    "fact_customer_activity", "fact_store_operations",
    "fact_weather", "fact_restriction_events",
]

existing_q = text("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'core'
""")

with engine.connect() as conn:
    existing = {r[0] for r in conn.execute(existing_q)}

for tbl in EXPECTED_TABLES:
    found = tbl in existing
    record(f"Table core.{tbl}", found, "" if found else "MISSING")

extra = existing - set(EXPECTED_TABLES)
if extra:
    print(f"\n  Extra tables in core schema (not checked): {sorted(extra)}")

---
## 3. Row Counts

In [ ]:
count_rows = []
for tbl in EXPECTED_TABLES:
    try:
        with engine.connect() as conn:
            n = conn.execute(text(f"SELECT COUNT(*) FROM core.{tbl}")).scalar()
        is_fact = tbl.startswith("fact")
        passed  = n >= MIN_ROWS if is_fact else n > 0
        record(f"Row count core.{tbl}", passed, f"{n:,} rows")
        count_rows.append({"table": tbl, "rows": n})
    except Exception as e:
        record(f"Row count core.{tbl}", False, str(e))
        count_rows.append({"table": tbl, "rows": 0})

counts_df = pd.DataFrame(count_rows)

fig, ax = plt.subplots(figsize=(12, 4))
colors = [PALETTE[0] if t.startswith("fact") else PALETTE[2] for t in counts_df["table"]]
ax.barh(counts_df["table"], counts_df["rows"], color=colors)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.set_xlabel("Row count")
ax.set_title("Row Counts per Table  (blue = fact, green = dim)")
plt.tight_layout(); plt.show()

---
## 4. Data Freshness

In [ ]:
DATE_TABLES = {
    "fact_sales":              "date",
    "fact_inventory":          "date",
    "fact_promotions":         "date",
    "fact_customer_activity":  "date",
    "fact_store_operations":   "date",
    "fact_weather":            "date",
    "fact_restriction_events": "start_date",
    "dim_calendar":            "date",
}

today = datetime.utcnow().date()
freshness_rows = []

for tbl, col in DATE_TABLES.items():
    try:
        with engine.connect() as conn:
            row = conn.execute(
                text(f"SELECT MIN({col}), MAX({col}) FROM core.{tbl}")
            ).fetchone()
        min_dt = pd.to_datetime(row[0]).date() if row[0] else None
        max_dt = pd.to_datetime(row[1]).date() if row[1] else None
        lag    = (today - max_dt).days if max_dt else None
        stale  = lag is None or lag > STALE_DAYS
        record(
            f"Freshness core.{tbl}",
            not stale,
            f"{min_dt} -> {max_dt}  (lag {lag}d)",
            warn=stale,
        )
        freshness_rows.append({"table": tbl, "min_date": min_dt,
                                "max_date": max_dt, "lag_days": lag})
    except Exception as e:
        record(f"Freshness core.{tbl}", False, str(e))
        freshness_rows.append({"table": tbl, "min_date": None,
                                "max_date": None, "lag_days": None})

display(pd.DataFrame(freshness_rows))

---
## 5. Null Check — Critical Columns

In [ ]:
NULL_CHECKS = {
    "fact_sales":     ["date", "store_id", "product_id", "units_sold", "price"],
    "dim_products":   ["product_id", "product_name", "category", "regular_price", "cost_price"],
    "dim_stores":     ["store_id", "store_name", "store_size"],
    "fact_promotions":["date", "store_id", "product_id", "promo_flag"],
    "dim_calendar":   ["date", "season", "month"],
}

for tbl, cols in NULL_CHECKS.items():
    for col in cols:
        try:
            with engine.connect() as conn:
                n_null = conn.execute(
                    text(f"SELECT COUNT(*) FROM core.{tbl} WHERE {col} IS NULL")
                ).scalar()
            record(f"No nulls in {tbl}.{col}", n_null == 0,
                   f"{n_null:,} nulls found" if n_null else "")
        except Exception as e:
            record(f"No nulls in {tbl}.{col}", False, str(e))

---
## 6. Margin Sanity — Price > Cost

In [ ]:
with engine.connect() as conn:
    inverted = conn.execute(text("""
        SELECT product_id, product_name, regular_price, cost_price
        FROM core.dim_products
        WHERE cost_price >= regular_price OR cost_price <= 0 OR regular_price <= 0
    """)).fetchall()

passed = len(inverted) == 0
record("Price > cost for all products", passed,
       f"{len(inverted)} product(s) with cost >= price" if not passed else "")

if inverted:
    cols = ["product_id","product_name","regular_price","cost_price"]
    display(pd.DataFrame(inverted, columns=cols))

# Margin distribution
df_prod = pd.read_sql("""
    SELECT product_id, category,
           (regular_price - cost_price) / NULLIF(regular_price, 0) * 100 AS margin_pct
    FROM core.dim_products
    WHERE regular_price > 0
""", engine)

fig, ax = plt.subplots(figsize=(10, 4))
for i, (cat, grp) in enumerate(df_prod.groupby("category")):
    ax.hist(grp["margin_pct"].dropna(), bins=30, alpha=0.6,
            label=cat, color=PALETTE[i % len(PALETTE)])
ax.axvline(15, color="red", ls="--", lw=1, label="15% floor")
ax.set_xlabel("Gross margin %")
ax.set_title("Product Margin Distribution by Category")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

---
## 7. Negative Values — units_sold and price

In [ ]:
checks_7 = [
    ("No negative units_sold",  "SELECT COUNT(*) FROM core.fact_sales WHERE units_sold < 0"),
    ("No zero/negative price",  "SELECT COUNT(*) FROM core.fact_sales WHERE price <= 0"),
    ("No negative inventory",   "SELECT COUNT(*) FROM core.fact_inventory WHERE stock_level < 0"),
]

for name, q in checks_7:
    try:
        with engine.connect() as conn:
            n = conn.execute(text(q)).scalar()
        record(name, n == 0, f"{n:,} rows" if n else "")
    except Exception as e:
        record(name, False, str(e))

---
## 8. Referential Integrity — Fact FK -> Dim

In [ ]:
fk_checks = [
    ("fact_sales.store_id -> dim_stores",
     "SELECT COUNT(*) FROM core.fact_sales f WHERE NOT EXISTS "
     "(SELECT 1 FROM core.dim_stores d WHERE d.store_id = f.store_id)"),

    ("fact_sales.product_id -> dim_products",
     "SELECT COUNT(*) FROM core.fact_sales f WHERE NOT EXISTS "
     "(SELECT 1 FROM core.dim_products d WHERE d.product_id = f.product_id)"),

    ("fact_sales.date -> dim_calendar",
     "SELECT COUNT(*) FROM core.fact_sales f WHERE NOT EXISTS "
     "(SELECT 1 FROM core.dim_calendar d WHERE d.date = f.date)"),

    ("fact_promotions.store_id -> dim_stores",
     "SELECT COUNT(*) FROM core.fact_promotions f WHERE NOT EXISTS "
     "(SELECT 1 FROM core.dim_stores d WHERE d.store_id = f.store_id)"),

    ("fact_promotions.product_id -> dim_products",
     "SELECT COUNT(*) FROM core.fact_promotions f WHERE NOT EXISTS "
     "(SELECT 1 FROM core.dim_products d WHERE d.product_id = f.product_id)"),
]

for name, q in fk_checks:
    try:
        with engine.connect() as conn:
            n = conn.execute(text(q)).scalar()
        record(name, n == 0, f"{n:,} orphan rows" if n else "")
    except Exception as e:
        record(name, False, str(e))

---
## 9. Promo discount_pct Bounds [0, 100]

In [ ]:
promo_checks = [
    ("discount_pct >= 0",   "SELECT COUNT(*) FROM core.fact_promotions WHERE discount_pct < 0"),
    ("discount_pct <= 100", "SELECT COUNT(*) FROM core.fact_promotions WHERE discount_pct > 100"),
    ("promo_flag in {0,1}", "SELECT COUNT(*) FROM core.fact_promotions WHERE promo_flag NOT IN (0, 1)"),
]

for name, q in promo_checks:
    try:
        with engine.connect() as conn:
            n = conn.execute(text(q)).scalar()
        record(f"Promo: {name}", n == 0, f"{n:,} invalid rows" if n else "")
    except Exception as e:
        record(f"Promo: {name}", False, str(e))

# Distribution of discount depths
df_promo = pd.read_sql("""
    SELECT discount_pct, promo_flag
    FROM core.fact_promotions
    WHERE promo_flag = 1 AND discount_pct > 0
""", engine)

if len(df_promo):
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.hist(df_promo["discount_pct"], bins=40, color=PALETTE[0], edgecolor="white")
    ax.set_xlabel("Discount %")
    ax.set_title(f"Promo Discount Depth Distribution  (n={len(df_promo):,} promo rows)")
    plt.tight_layout(); plt.show()

---
## 10. Summary Dashboard

In [ ]:
summary = pd.DataFrame(results)
n_pass  = (summary["status"] == "PASS").sum()
n_warn  = (summary["status"] == "WARN").sum()
n_fail  = (summary["status"] == "FAIL").sum()
total   = len(summary)

print("=" * 55)
print(f"  HEALTH CHECK COMPLETE -- {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 55)
print(f"  Passed : {n_pass:>3} / {total}")
print(f"  Warned : {n_warn:>3} / {total}")
print(f"  Failed : {n_fail:>3} / {total}")
print("=" * 55)

if n_fail:
    print("\n  FAILED checks:")
    display(summary[summary["status"] == "FAIL"][["check","detail"]])

if n_warn:
    print("\n  WARNINGS:")
    display(summary[summary["status"] == "WARN"][["check","detail"]])

# ── Visual scorecard ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

pie_vals   = [n_pass, n_warn, n_fail]
pie_labels = [f"Pass ({n_pass})", f"Warn ({n_warn})", f"Fail ({n_fail})"]
pie_colors = [PALETTE[2], PALETTE[3], PALETTE[1]]
axes[0].pie([v for v in pie_vals if v > 0],
            labels=[l for v, l in zip(pie_vals, pie_labels) if v > 0],
            colors=[c for v, c in zip(pie_vals, pie_colors) if v > 0],
            autopct="%1.0f%%", startangle=90)
axes[0].set_title("Check Results")

color_map = {"PASS": PALETTE[2], "WARN": PALETTE[3], "FAIL": PALETTE[1]}
bar_colors = [color_map[s] for s in summary["status"]]
axes[1].barh(summary["check"], [1]*len(summary), color=bar_colors, height=0.6)
axes[1].set_xlim(0, 1.2)
axes[1].set_xticks([])
axes[1].tick_params(axis="y", labelsize=7)
axes[1].set_title("Per-Check Status")
axes[1].invert_yaxis()

plt.tight_layout(); plt.show()

# ── Spot stats ────────────────────────────────────────────
print("\n-- Quick stats from fact_sales --")
spot = pd.read_sql("""
    SELECT
        COUNT(*)                          AS total_transactions,
        COUNT(DISTINCT store_id)          AS stores,
        COUNT(DISTINCT product_id)        AS products,
        MIN(date)                         AS earliest_date,
        MAX(date)                         AS latest_date,
        ROUND(AVG(units_sold)::numeric,2) AS avg_units_per_txn,
        ROUND(AVG(price)::numeric,2)      AS avg_price
    FROM core.fact_sales
""", engine)
display(spot.T.rename(columns={0: "value"}))